# Decision Tree (Pipeline B)

**Model**: `DecisionTreeRegressor` / **Target**: gdpc1  
**Variables**: Cat3 (53 + COVID = 56) / **n_lags**: 4  
**Scaling**: NO / **HP tuning**: YES / **10-model averaging**: YES

In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from scipy.stats import norm
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]
sys.path.insert(0, os.path.join(os.path.pardir, 'data'))
from helpers import (gen_lagged_data, gen_vintage_data, make_supervised_vintage_frame, flatten_data, mean_fill_dataset,
                     get_features, load_data, PREDICTIONS_DIR, TARGET)
SEED = 42; np.random.seed(SEED)
TRAIN_START = '1959-01-01'; TRAIN_END = '2007-12-31'; VAL_END = '2016-12-31'
TEST_START = '2017-01-01'; TEST_END = '2025-12-31'
N_LAGS = 4; N_MODELS = 10; VINTAGES = {'m1': -2, 'm2': -1, 'm3': 0, 'post1': 1}
print('DT — Cat3, n_lags=4')

DT — Cat3, n_lags=4


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
# Phase A: HP tuning
tune = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= VAL_END)].reset_index(drop=True)
tune_f = mean_fill_dataset(tune, tune)
tune_fl = flatten_data(tune_f, TARGET, N_LAGS)
tune_fl = tune_fl.loc[tune_fl.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how='any').reset_index(drop=True)
fcols = [c for c in tune_fl.columns if c != 'date' and c != TARGET and any(c == f or c.startswith(f + '_') for f in features)]
tscv = TimeSeriesSplit(n_splits=5)
dt_grid = {'max_depth': [None, 5, 10, 20], 'min_samples_leaf': [1, 3, 5]}
gs = GridSearchCV(DecisionTreeRegressor(criterion='squared_error', random_state=SEED),
    dt_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=1)
gs.fit(tune_fl[fcols].values, tune_fl[TARGET].values)
bp = gs.best_params_
print(f'Best DT params: {bp}')

Best DT params: {'max_depth': None, 'min_samples_leaf': 5}


In [3]:
test_quarters = monthly[(monthly['date'] >= TEST_START) & (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3,6,9,12])]['date'].tolist()
predictions = {v: [] for v in VINTAGES}; actuals_list = []; failed = 0
for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly['date'] == pd_q][TARGET].values[0])
    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        train_fl, tr_fl, _ = make_supervised_vintage_frame(
            metadata, monthly, TARGET, features, TRAIN_START, pd_q, fc, N_LAGS
        )
        if len(train_fl) < 10:
            predictions[vn].append(np.nan); continue
        try:
            models = []
            for k in range(N_MODELS):
                m = DecisionTreeRegressor(criterion='squared_error', max_depth=bp['max_depth'],
                    min_samples_leaf=bp['min_samples_leaf'], random_state=SEED+k)
                m.fit(train_fl[fcols].values, train_fl[TARGET].values)
                models.append(m)
            preds = [m.predict(tr_fl[fcols].values)[0] for m in models]
            predictions[vn].append(np.nanmean(preds))
        except Exception:
            predictions[vn].append(np.nan); failed += 1
    if (i+1) % 8 == 0:
        print(f'  {i+1}/{len(test_quarters)}')
print(f'Done. {failed} failures.')

  8/36


  16/36


  24/36


  32/36


Done. 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({'date': test_quarters, 'actual': actuals_list, 'prediction': predictions[vn]}).to_csv(
        os.path.join(PREDICTIONS_DIR, f'dt_{vn}.csv'), index=False)
    print(f'Saved dt_{vn}.csv')

Saved dt_m1.csv
Saved dt_m2.csv
Saved dt_m3.csv
Saved dt_post1.csv


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum() > 0 else np.nan
def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum() > 0 else np.nan

panels = {
    'Pre-COVID  (2017-2019)': ('2017-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2017-2025)': ('2017-01-01', '2025-12-31'),
}
a = np.array(actuals_list); d = pd.to_datetime(test_quarters)
print(f'DT best params: {bp}')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print(f'--- {pn} ---')
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print(f'  {vn}  RMSFE={rmse(a[m], pp[m]):.6f}  MAE={mae(a[m], pp[m]):.6f}')

DT best params: {'max_depth': None, 'min_samples_leaf': 5}
--- Pre-COVID  (2017-2019) ---
  m1  RMSFE=0.002519  MAE=0.002096
  m2  RMSFE=0.002535  MAE=0.002056
  m3  RMSFE=0.002726  MAE=0.002311
  post1  RMSFE=0.002938  MAE=0.002497
--- COVID      (2020-2021) ---
  m1  RMSFE=0.042451  MAE=0.026573
  m2  RMSFE=0.042460  MAE=0.026624
  m3  RMSFE=0.042303  MAE=0.026417
  post1  RMSFE=0.042159  MAE=0.026337
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.004600  MAE=0.003425
  m2  RMSFE=0.004540  MAE=0.003377
  m3  RMSFE=0.004685  MAE=0.003531
  post1  RMSFE=0.004684  MAE=0.003524
--- Full test  (2017-2025) ---
  m1  RMSFE=0.019329  MAE=0.007957
  m2  RMSFE=0.019327  MAE=0.007933
  m3  RMSFE=0.019265  MAE=0.008028
  post1  RMSFE=0.019279  MAE=0.008131
